# This file will calculate the straight-line distance between properties and CBD.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from geopy.distance import great_circle

In [2]:
# read the files record propertry
property_data_origin = pd.read_csv('../data/curated/external_data/combined_cloest_station.csv')
property_data_origin

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,school_distance(KM),closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM),closest_distance_station(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,1.898006,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903,"[-38.13958090963712, 145.36252669303045]",3.840116,"['[-38.05104856543076, 145.3665303282505]', '[...",6.090210,11.8866
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,0.705427,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025,"[-38.03401014838179, 145.37166338662394]",5.566924,"['[-38.05104856543076, 145.3665303282505]', '[...",3.619981,5.3735
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,...,1.100561,"[-38.045325, 145.347181]",6.344909,"[-38.0604189, 145.3394612]",5.332842,"[-38.13958090963712, 145.36252669303045]",5.064419,"['[-38.0663192834778, 145.4116572812351]', '[-...",4.320648,8.9170
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,...,0.674921,"[-38.113312, 145.280832]",5.678594,"[-38.1184718, 145.3213262]",2.210922,"[-38.13958090963712, 145.36252669303045]",3.267265,"['[-38.0995872148084, 145.2804738077291]', '[-...",5.911467,8.1066
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,...,2.225124,"[-38.113312, 145.280832]",7.522231,"[-38.1184718, 145.3213262]",3.956745,"[-38.13958090963712, 145.36252669303045]",1.122928,"['[-38.0995872148084, 145.2804738077291]', '[-...",8.056215,10.1328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,...,0.107655,"[-37.800508, 144.895155]",1.454065,"[-37.8016476, 144.8977057]",1.411290,"[-37.83084513979606, 144.91345002848863]",2.791844,"['[-37.81567190405462, 144.89006145414797]', '...",0.290453,1.1480
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,...,0.239821,"[-37.797407, 144.887421]",3.072331,"[-37.828989, 144.84627]",2.396280,"[-37.83084513979606, 144.91345002848863]",3.746179,"['[-37.83072411396968, 144.88584876958876]', '...",1.522703,2.1264
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,...,0.819385,"[-37.797407, 144.887421]",2.789294,"[-37.828989, 144.84627]",2.273966,"[-37.83084513979606, 144.91345002848863]",4.413664,"['[-37.7993395010312, 144.86344949623717]', '[...",1.915810,2.8147
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,...,0.290245,"[-37.797407, 144.887421]",1.865866,"[-37.8016476, 144.8977057]",2.108881,"[-37.83084513979606, 144.91345002848863]",3.729966,"['[-37.81567190405462, 144.89006145414797]', '...",1.122803,1.8488


In [3]:
# only select the needed features for running faster
property_data =  property_data_origin[['name','coordinates']]
property_data.head()

,name,coordinates
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]"
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]"
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]"
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]"
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]"


In [4]:
# create a DataFrame containing melbourne_CBD_coord
CBD_data = {'melbourne_CBD_coord': ['[-37.48490608, 144.57470088]']}
CBD_df = pd.DataFrame(CBD_data)
# save as CSV
#df.to_csv('melbourne_CBD_coord.csv', index=False)
CBD_df

,melbourne_CBD_coord
0,"[-37.48490608, 144.57470088]"


In [5]:
# This function will accept a string to parse coordinate string into point object
def parse_coordinates(coord_str):
    parts = coord_str.strip('[]').split(',')
    return (float(parts[0]), float(parts[1]))

In [6]:
# parse coordinate string into point object for both property & CBD data
closest_CBD = []
CBD_distance = []
property_coordinates = property_data['coordinates'].apply(parse_coordinates)
all_CBD_coordinates = CBD_df['melbourne_CBD_coord'].apply(parse_coordinates)

# find the closest CBD for each property
for property_coord in property_coordinates:
    # By default: no nearest CBD and the distance is positive infinity.
    min_distance = float('inf')
    closest_CBD_geo = None
    
    for i, CBD_coord in enumerate(all_CBD_coordinates):
        # check if the value of the coordinate point is valid
        if np.isnan(property_coord).any() or np.isnan(CBD_coord).any():
            continue
        # claculate the distance between property and CBD, update if it is smallest
        distance = great_circle(property_coord, CBD_coord).kilometers
        if distance < min_distance:
            min_distance = distance
            closest_CBD_geo = CBD_df.loc[i, 'melbourne_CBD_coord']

    # add feature to store closest CBD for the property
    closest_CBD.append(closest_CBD_geo)
    CBD_distance.append(min_distance)
#property_data['closest_CBD'] = closest_CBD
property_data['CBD_distance(KM)'] = CBD_distance
property_data

/tmp/ipykernel_7452/2243467969.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  property_data['CBD_distance(KM)'] = CBD_distance


,name,coordinates,CBD_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]",97.390646
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]",95.567693
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]",98.122824
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]",97.325206
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]",99.752000
...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]",45.903694
8798,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]",45.752205
8799,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]",44.935453
8800,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]",45.157145


In [7]:
# store closest CBD information to the merged data
property_data_origin['CBD_distance(KM)'] = CBD_distance
# save as a CSV file
property_data_origin.to_csv("../data/curated/merged_data/merged_data_with_facility.csv", index=False)
merged_data_with_facility = pd.read_csv("../data/curated/merged_data/merged_data_with_facility.csv")
merged_data_with_facility

,name,rental_price,num_bedroom,num_bathroom,num_parking,postcode,coordinates,property_geometry,sa2_code,sa2_name,...,closest_hospital,hospital_distance(KM),closest_mall,mall_distance(KM),closest_park,park_distance(KM),closest_station,station_distance(KM),closest_distance_station(KM),CBD_distance(KM)
0,31 Chittagong Drive Clyde North VIC 3978,575.0,4,2,2.0,3978.0,"[-38.1053122, 145.3570863]",POINT (145.3570863 -38.1053122),212031556.0,Clyde North - South,...,"[-38.045325, 145.347181]",6.726397,"[-38.1184718, 145.3213262]",3.453903,"[-38.13958090963712, 145.36252669303045]",3.840116,"['[-38.05104856543076, 145.3665303282505]', '[...",6.090210,11.8866,97.390646
1,50 Elmtree Crescent Clyde North VIC 3978,560.0,4,2,2.0,3978.0,"[-38.0825712, 145.3561984]",POINT (145.3561984 -38.0825712),212031555.0,Clyde North - North,...,"[-38.045325, 145.347181]",4.216162,"[-38.0604189, 145.3394612]",2.866025,"[-38.03401014838179, 145.37166338662394]",5.566924,"['[-38.05104856543076, 145.3665303282505]', '[...",3.619981,5.3735,95.567693
2,7 Mortdale Lane Clyde North VIC 3978,490.0,2,2,1.0,3978.0,"[-38.0961758, 145.3800644]",POINT (145.3800644 -38.0961758),212031556.0,Clyde North - South,...,"[-38.045325, 145.347181]",6.344909,"[-38.0604189, 145.3394612]",5.332842,"[-38.13958090963712, 145.36252669303045]",5.064419,"['[-38.0663192834778, 145.4116572812351]', '[-...",4.320648,8.9170,98.122824
3,54 Walhallow Drive Clyde North VIC 3978,540.0,4,2,1.0,3978.0,"[-38.1133324, 145.3457396]",POINT (145.3457396 -38.1133324),212031556.0,Clyde North - South,...,"[-38.113312, 145.280832]",5.678594,"[-38.1184718, 145.3213262]",2.210922,"[-38.13958090963712, 145.36252669303045]",3.267265,"['[-38.0995872148084, 145.2804738077291]', '[-...",5.911467,8.1066,97.325206
4,10 Sicily Road Clyde North VIC 3978,520.0,4,2,2.0,3978.0,"[-38.1295789, 145.3642993]",POINT (145.3642993 -38.1295789),212031303.0,Cranbourne South,...,"[-38.113312, 145.280832]",7.522231,"[-38.1184718, 145.3213262]",3.956745,"[-38.13958090963712, 145.36252669303045]",1.122928,"['[-38.0995872148084, 145.2804738077291]', '[-...",8.056215,10.1328,99.752000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8797,61 Tongue Street Yarraville VIC 3013,630.0,2,1,0.0,3013.0,"[-37.8131463, 144.8909053]",POINT (144.8909053 -37.8131463),213031352.0,Yarraville,...,"[-37.800508, 144.895155]",1.454065,"[-37.8016476, 144.8977057]",1.411290,"[-37.83084513979606, 144.91345002848863]",2.791844,"['[-37.81567190405462, 144.89006145414797]', '...",0.290453,1.1480,45.903694
8798,47 Mill Avenue Yarraville VIC 3013,730.0,4,3,2.0,3013.0,"[-37.8222822, 144.872198]",POINT (144.872198 -37.8222822),213031352.0,Yarraville,...,"[-37.797407, 144.887421]",3.072331,"[-37.828989, 144.84627]",2.396280,"[-37.83084513979606, 144.91345002848863]",3.746179,"['[-37.83072411396968, 144.88584876958876]', '...",1.522703,2.1264,45.752205
8799,12 Adeney Street Yarraville VIC 3013,450.0,3,1,2.0,3013.0,"[-37.8163817, 144.8666543]",POINT (144.8666543 -37.8163817),213031352.0,Yarraville,...,"[-37.797407, 144.887421]",2.789294,"[-37.828989, 144.84627]",2.273966,"[-37.83084513979606, 144.91345002848863]",4.413664,"['[-37.7993395010312, 144.86344949623717]', '[...",1.915810,2.8147,44.935453
8800,229B Somerville Road Yarraville VIC 3013,300.0,1,1,0.0,3013.0,"[-37.8124289, 144.8779569]",POINT (144.8779569 -37.8124289),213031352.0,Yarraville,...,"[-37.797407, 144.887421]",1.865866,"[-37.8016476, 144.8977057]",2.108881,"[-37.83084513979606, 144.91345002848863]",3.729966,"['[-37.81567190405462, 144.89006145414797]', '...",1.122803,1.8488,45.157145
